### Loading and Exploring the Dataset

In [54]:
import pandas as pd
import numpy as np

# 1. Load the raw dataset
df = pd.read_csv('02_expanded_dirty_data.csv')

# 2. Print basic info about the dataset
print("Dataset loaded successfully.")
print("Original row count:", len(df))
print("Original column count:", df.shape[1])

# 3. Check data types and structure
print(df.info())

# 4. Check for any missing values
print(df.isna().sum())

Dataset loaded successfully.
Original row count: 250000
Original column count: 23
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 23 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   UserID                         250000 non-null  object 
 1   Age                            239064 non-null  float64
 2   Gender                         240667 non-null  object 
 3   Country                        236579 non-null  object 
 4   Device                         245008 non-null  object 
 5   GameGenre                      245017 non-null  object 
 6   SessionCount                   245049 non-null  float64
 7   AverageSessionLength           240232 non-null  object 
 8   SpendingSegment                250000 non-null  object 
 9   InAppPurchaseAmount            243747 non-null  object 
 10  FirstPurchaseDaysAfterInstall  210424 non-null  float64
 11  PaymentMe

### Removing Duplicate Rows

In [55]:
# Check how many duplicate rows exist in the dataset
duplicates_count = df.duplicated().sum()

# Drop the duplicate rows to keep only unique data
df = df.drop_duplicates()

# Print the results to see how many duplicates were removed
print("Duplicates found and removed:", duplicates_count)
print("Row count after removing duplicates:", len(df))

Duplicates found and removed: 2500
Row count after removing duplicates: 247500


### Cleaning the InAppPurchaseAmount Column

In [56]:
# Count how many rows contain the dollar sign symbol
dollar_count = df['InAppPurchaseAmount'].astype(str).str.contains(r'\$').sum()

# Convert the column to text, remove the '$' sign, and convert back to numbers
df['InAppPurchaseAmount'] = df['InAppPurchaseAmount'].astype(str).str.replace('$', '', regex=False)
df['InAppPurchaseAmount'] = pd.to_numeric(df['InAppPurchaseAmount'], errors='coerce')

# Print a confirmation message
print("Cleaned 'InAppPurchaseAmount': removed '$' from", dollar_count, "rows and converted to numeric.")

Cleaned 'InAppPurchaseAmount': removed '$' from 2444 rows and converted to numeric.


### Cleaning the AverageSessionLength Column

In [57]:
# Count how many rows contain the text 'sec'
sec_count = df['AverageSessionLength'].astype(str).str.contains('sec').sum()

# Convert to text, remove 'sec', and convert the remaining text into numbers
df['AverageSessionLength'] = df['AverageSessionLength'].astype(str).str.replace('sec', '', regex=False)
df['AverageSessionLength'] = pd.to_numeric(df['AverageSessionLength'], errors='coerce')

# Print a confirmation message
print("Cleaned 'AverageSessionLength': removed 'sec' text from", sec_count, "rows and converted to numeric.")

Cleaned 'AverageSessionLength': removed 'sec' text from 2929 rows and converted to numeric.


### Standardizing Date Formats

In [58]:
# Convert date columns into standard Python datetime format so they are uniform
df['InstallDate'] = pd.to_datetime(df['InstallDate'], errors='coerce')
df['LastPurchaseDate'] = pd.to_datetime(df['LastPurchaseDate'], errors='coerce')

# Print confirmation that dates are standardized
print("Converted 'InstallDate' and 'LastPurchaseDate' to standard datetime format.")

Converted 'InstallDate' and 'LastPurchaseDate' to standard datetime format.


### Saving the Cleaned File

In [59]:
# Define the name of the output file
output_file = 'the_clean_data_part1.csv'

# Save the cleaned DataFrame into a new CSV file
df.to_csv(output_file, index=False)

# Print final success message and the final dataset shape
print("Part 1 is done! Saved the cleaned file as:", output_file)
print("Final dataset shape:", df.shape)

Part 1 is done! Saved the cleaned file as: the_clean_data_part1.csv
Final dataset shape: (247500, 23)


### **Cleaning Part 2** - Text Standardization, Missing Values & Outliers

### Standardize categorical text labels

In [60]:
# 1. Standardize categorical text labels (fix case sensitivity & extra spaces)
categorical_cols_to_standardize = [
    'Device',
    'GameGenre',
    'PaymentMethod',
    'AcquisitionChannel',
    'GuildMember',
    'PushNotificationsOptIn',
]

for col in categorical_cols_to_standardize:
  if col in df.columns:
    df[col] = df[col].astype(str).str.strip().str.title()
    df[col] = df[col].replace('Nan', 'Unknown')

print('Standardized categorical text labels.')

Standardized categorical text labels.


### Handle missing values

In [61]:
# a) Replace missing numeric values with median
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
  missing_count = df[col].isnull().sum()
  if missing_count > 0:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(
        f'Replaced {missing_count} missing values in numeric column "{col}" with'
        f' median: {median_val}'
    )

Replaced 10825 missing values in numeric column "Age" with median: 33.0
Replaced 4883 missing values in numeric column "SessionCount" with median: 10.0
Replaced 9769 missing values in numeric column "AverageSessionLength" with median: 20.11
Replaced 6181 missing values in numeric column "InAppPurchaseAmount" with median: 11.37
Replaced 39192 missing values in numeric column "FirstPurchaseDaysAfterInstall" with median: 16.0
Replaced 3024 missing values in numeric column "ActiveDays30d" with median: 6.0
Replaced 3024 missing values in numeric column "PlayerLevel30d" with median: 12.0
Replaced 3024 missing values in numeric column "AdsWatched30d" with median: 5.0
Replaced 3024 missing values in numeric column "SocialActions30d" with median: 4.0
Replaced 3024 missing values in numeric column "PurchaseCount30d" with median: 1.0
Replaced 3024 missing values in numeric column "WinRate" with median: 0.4


In [62]:
# b) Replace missing text/categorical values with 'Unknown'
text_cols = df.select_dtypes(include=['object']).columns
for col in text_cols:
  missing_count = df[col].isnull().sum()
  if missing_count > 0:
    df[col] = df[col].fillna('Unknown')
    print(
        f'Replaced {missing_count} missing values in text column "{col}" with'
        " 'Unknown'"
    )

Replaced 9241 missing values in text column "Gender" with 'Unknown'
Replaced 13309 missing values in text column "Country" with 'Unknown'


### outliers

In [63]:
# a) Convert negative numbers in Age or SessionCount to positive using .abs()
for col in ["Age", "SessionCount"]:
  if col in df.columns:
    negative_count = (df[col] < 0).sum()
    if negative_count > 0:
      df[col] = df[col].abs()
      print(f'Fixed {negative_count} negative values in "{col}" using abs().')

Fixed 733 negative values in "Age" using abs().
Fixed 488 negative values in "SessionCount" using abs().


In [64]:
# b) Fix unrealistic future dates in 'InstallDate'
if "InstallDate" in df.columns:
  future_dates_mask = df["InstallDate"] > pd.to_datetime("2026-11-13")
  if future_dates_mask.sum() > 0:
    df.loc[future_dates_mask, "InstallDate"] = pd.to_datetime("2026-01-01")
    print(
        f'Fixed {future_dates_mask.sum()} unrealistic future dates in'
        ' "InstallDate".'
    )

Fixed 1 unrealistic future dates in "InstallDate".


### Save the Final Cleaned Dataset

In [65]:
final_output_file = "03_clean_data.csv"
df.to_csv(final_output_file, index=False)

print(
    "Part 2 complete! Final cleaned dataset saved as:"
    f" '{final_output_file}'"
)
print("Final dataset shape:", df.shape)
print("Total missing values remaining:", df.isna().sum().sum())

Part 2 complete! Final cleaned dataset saved as: '03_clean_data.csv'
Final dataset shape: (247500, 23)
Total missing values remaining: 59359
